# Разведочный анализ

Обучающие таблицы соревнования: 2 991 396 пар и 3 344 613 объявлений.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

processed = Path("../data/processed")

In [2]:
pairs = pd.read_parquet(processed / "pairs.parquet")
pairs.shape

(2991396, 4)

In [3]:
pairs.dtypes

itemID_1            int64
itemID_2            int64
isDuplicate         int64
generationMethod    int64
dtype: object

## Баланс классов и способ разметки

Сколько дублей среди пар и как их доля зависит от способа разметки.

In [4]:
pairs["isDuplicate"].value_counts()

isDuplicate
0    1727117
1    1264279
Name: count, dtype: int64

In [5]:
pairs["isDuplicate"].mean()

np.float64(0.4226384604378691)

In [6]:
pairs.groupby("generationMethod")["isDuplicate"].agg(["count", "mean"])

,count,mean
generationMethod,,
1,2041604,0.508611
2,141914,1.000000
3,807878,0.103955


Дублей 42,3 %. По способам разметки доля сильно разная:

| generationMethod | Пар | Доля дублей |
|---|---|---|
| 1 | 2 041 604 | 50,9 % |
| 2 | 141 914 | 100 % |
| 3 | 807 878 | 10,4 % |

Колонка почти выдаёт ответ. В признаки она не попадает, но подходит для разбора метрик по способам.

## Проверка пар

Нет ли повторов пар, пар из одного и того же объявления и зеркальных пар.

In [7]:
pairs.duplicated(["itemID_1", "itemID_2"]).sum()

np.int64(0)

In [8]:
(pairs["itemID_1"] < pairs["itemID_2"]).mean()

np.float64(1.0)

In [9]:
(pairs["itemID_1"] == pairs["itemID_2"]).sum()

np.int64(0)

Ничего из этого нет. Первый ID всегда меньше второго, так что зеркальных пар быть не может.

## Пропуски в объявлениях

Доля пропусков по колонкам таблицы объявлений.

In [10]:
path = processed / "items.parquet"
columns = [
    "categoryID",
    "title",
    "description",
    "images_array",
    "attrsJSON",
    "price",
    "locationID",
    "metroID",
    "lat",
    "lon",
]
for column in columns:
    share = pd.read_parquet(path, columns=[column])[column].isna().mean()
    print(f"{column}: {share:.3f}")

categoryID: 0.000
title: 0.000
description: 0.000
images_array: 0.119
attrsJSON: 0.031
price: 0.096
locationID: 0.000
metroID: 0.671
lat: 0.000
lon: 0.000


Метро не указано у 67 % объявлений, фотографий нет у 12 %, цены нет у 9,6 %, атрибутов нет у 3,1 %. В остальных колонках пропусков нет.

## Группы объявлений

Пары связывают объявления в цепочки: из пар (A, B) и (B, C) получается группа из A, B и C. Смотрим, какого размера получаются группы.

In [11]:
codes, uniques = pd.factorize(pd.concat([pairs["itemID_1"], pairs["itemID_2"]]))
n_pairs = len(pairs)
left = codes[:n_pairs]
right = codes[n_pairs:]
len(uniques)

3344613

In [12]:
graph = coo_matrix(
    (np.ones(n_pairs, dtype=np.int8), (left, right)),
    shape=(len(uniques), len(uniques)),
)
n_groups, labels = connected_components(graph, directed=False)
n_groups

1253680

In [13]:
sizes = pd.Series(labels).value_counts()
sizes.describe()

count    1.253680e+06
mean     2.667836e+00
std      3.562961e+00
min      2.000000e+00
25%      2.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      9.730000e+02
Name: count, dtype: float64

In [14]:
sizes.head(10)

457      973
329      712
6739     636
5375     588
24784    585
1968     583
3672     488
3066     470
2374     453
9285     442
Name: count, dtype: int64

In [15]:
pair_group = labels[left]
pairs_per_group = pd.Series(pair_group).value_counts()
pairs_per_group.head(5) / n_pairs

457     0.000404
329     0.000386
2374    0.000386
6739    0.000355
9285    0.000352
Name: count, dtype: float64

Всего 1 253 680 групп. Типичная группа состоит из двух объявлений, самая большая из 973. Огромных групп нет: самая большая даёт около 0,04 % всех пар.


## Утечка при случайном разбиении

В проверку отправляется около 20 % пар. Считаем, у какой доли проверочных пар хотя бы одно объявление встречается и в обучающих парах.

In [16]:
def leaked_share(is_valid):
    train_ads = np.zeros(len(uniques), dtype=bool)
    train_ads[left[~is_valid]] = True
    train_ads[right[~is_valid]] = True
    shared = train_ads[left[is_valid]] | train_ads[right[is_valid]]
    return shared.mean()

In [17]:
rng = np.random.default_rng(42)
random_valid = rng.random(n_pairs) < 0.2
leaked_share(random_valid)

np.float64(0.6587911858344963)

In [18]:
group_valid = rng.random(n_groups) < 0.2
leaked_share(group_valid[pair_group])

np.float64(0.0)

In [19]:
group_valid[pair_group].mean()

np.float64(0.20167941656671334)

При случайном разбиении пар у 65,9 % проверочных пар хотя бы одно объявление уже встречалось в обучающих парах. При разбиении по группам таких пар нет, а в проверку попадает 20,2 % данных. Для валидации берётся разбиение по группам.